# Core "Advanced Hybrid Coupling" Go-Live, 2026-06-10/11

Historical reproduction (see the [bertellis-flow-diagnostics](../bertellis-flow-diagnostics/bertellis_flow_diagnostics.ipynb) notebook) only proves a platform can look backward at known moments. This notebook points the same reusable functions &ndash; `hourly_border_counts`, `before_after_convergence`, `convergence_summary` &ndash; at a real, ongoing structural change: a market-coupling methodology change on Core CCR's external borders. ([JAO confirmation](https://www.jao.eu/news/confirmation-core-advanced-hybrid-coupling-go-live))

The short window is 8 days (event date &plusmn; 4 days) at hourly resolution &ndash; wide enough to see a trend, narrow enough that the API doesn't auto-downgrade past hourly.

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai) (docs at [docs.voltpath.ai](https://docs.voltpath.ai)). Physical flow data is CC-BY 4.0; day-ahead prices are reused under ENTSO-E's general Transparency Platform terms.

**Before you run this:** sign up at [voltpath.ai](https://voltpath.ai) for a free personal API key (copy it any time from [voltpath.ai/settings](https://voltpath.ai/settings)), then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY` (or leave `.env` empty and you'll be prompted).


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent))

from common import (
    FLOW_COLORS,
    before_after_convergence,
    before_after_convergence_chart,
    border_ccr_mechanism,
    congestion_ci_chart,
    convergence_summary,
    fetch_flows,
    save_chart,
)

In [ ]:
CORE_AHC_EVENT = {
    "name": "Core Advanced Hybrid Coupling",
    "slug": "core-ahc",
    "date": "2026-06-10",
    "date_from": "2026-06-06",
    "date_to": "2026-06-14",
}

### Congested / counter-intuitive borders

In [ ]:
core_ahc_event = CORE_AHC_EVENT
core_ahc_df = fetch_flows(core_ahc_event["date_from"], core_ahc_event["date_to"], resolution="1H")
fig = congestion_ci_chart(core_ahc_df, core_ahc_event)
save_chart(
    fig,
    f"{core_ahc_event['slug']}-congestion-ci-{core_ahc_event['date_from']}-to-{core_ahc_event['date_to']}",
)
fig.show()

### Did coupling reliability actually change?

In [ ]:
comparison = before_after_convergence(core_ahc_df, core_ahc_event["date"])
fig = before_after_convergence_chart(comparison, core_ahc_event)
save_chart(
    fig,
    f"{core_ahc_event['slug']}-convergence-before-after-{core_ahc_event['date_from']}-to-{core_ahc_event['date_to']}",
)
fig.show()

### Real effect or definitional artifact? And did it recover?

Advanced Hybrid Coupling changes how Core's *external* borders (interconnections to non-flow-based neighbouring CCRs, e.g. Core&ndash;Nordic, Core&ndash;Baltic, Core&ndash;Switzerland) are modelled: they go from a static D-2 exchange forecast to a virtual bidding zone solved jointly with the rest of the Core flow-based domain ([Sch&ouml;nheit & Marjanovi&#263;, arXiv:2405.10010](https://arxiv.org/abs/2405.10010); [JAO go-live notice](https://www.jao.eu/news/go-live-date-core-advanced-hybrid-coupling)). JAO's own KPI Reading Guide defines the convergence KPI purely from clearing-price differences between neighbouring bidding zones, independent of how the underlying domain is solved &ndash; which argues against "the metric's definition just changed" and for "AHC genuinely reallocates capacity differently." JAO also flagged concrete data-publication defects (inaccurate RAM, missing scheduled exchanges/ATCs) on exactly the newly-affected borders in the 48 hours after go-live ([JAO: successful go-live notice](https://www.jao.eu/news/successful-go-live-core-advanced-hybrid-coupling)), a plausible source of first-days noise.

There's a direct precedent: Core's 2022 flow-based go-live produced a measured, temporary convergence dip (5.10% &rarr; 4.02%) that later recovered. The chart below checks whether AHC's external Core borders show the same transitional-dip-then-recovery pattern, using `border_ccr_mechanism` (already in `common/regions.py`) to isolate exactly the borders AHC's methodology change actually touches, rather than diluting the signal across all of Core's mostly-unaffected internal borders.


In [ ]:
CORE_ZONES = ["AT", "CZ", "DE_LU", "HU", "SI", "BE", "FR", "NL", "HR", "RO", "SK", "PL", "IT_NORD"]


def is_core_external_border(border: str) -> bool:
    zone_from, zone_to = border.split(" -> ")
    touches_core = zone_from in CORE_ZONES or zone_to in CORE_ZONES
    return touches_core and border_ccr_mechanism(border) != "Core (price-coupled, flow-based)"


# A 13-zone filter touches ~44 borders - wide enough that /api/v1/flows's
# response-size cap (exactly 20,000 rows) silently truncates the reply instead
# of erroring, cutting off the tail of the requested range (verified: even an
# 8-month chunk over this zone set still hit the 20,000-row cap). Monthly
# chunks stay comfortably under it; each is still cached individually by
# fetch_flows.
chunk_starts = pd.date_range("2025-06-01", "2026-09-24", freq="MS")
chunk_bounds = list(chunk_starts) + [pd.Timestamp("2026-09-24")]
core_ahc_long = pd.concat(
    [
        fetch_flows(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"), resolution="1D", zone=CORE_ZONES)
        for start, end in zip(chunk_bounds[:-1], chunk_bounds[1:])
    ],
    ignore_index=True,
)
core_ahc_long = core_ahc_long[core_ahc_long["border"].map(is_core_external_border)]

monthly_convergence_ahc = (
    core_ahc_long.set_index("datetime")
    .groupby(pd.Grouper(freq="MS"))
    .apply(lambda g: convergence_summary(g)["share_converged"].mean())
    .rename("share_converged")
    .reset_index()
)

fig = px.line(
    monthly_convergence_ahc,
    x="datetime",
    y="share_converged",
    title="Core external borders: monthly avg. share_converged, Jun 2025 to Sep 2026",
    labels={"share_converged": "Avg. share converged", "datetime": "Month"},
    color_discrete_sequence=[FLOW_COLORS["Convergence"]],
)
fig.add_vline(x=core_ahc_event["date"], line_dash="dash", annotation_text="Core AHC go-live")
save_chart(fig, "core-ahc-monthly-convergence-external-borders-2025-06-to-2026-09")
fig.show()

Corrected ACER-decision citation and the go-live date discrepancy found in TSO meeting minutes were checked during research for this section.

## Takeaways

- The same functions used to reproduce a year-old published analysis (see [bertellis-flow-diagnostics](../bertellis-flow-diagnostics/bertellis_flow_diagnostics.ipynb)) turn a historical curiosity into a live monitoring tool: point them at a market-coupling methodology change and the before/after shift shows up mechanically.
- Core AHC is a case where congestion and convergence decouple &ndash; a market-algorithm change, not a grid-topology change &ndash; unlike a physical event like Baltic sync, where the two metrics move together. Tracking both is the point: they answer different "why" questions.

---

Data: [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai).
